In [1]:
from ET_pylib.setup import *
setup_notebook()

# Kernekraftstød: numerisk analyse

Tabeller for basisforløbet mod kernekraftforløbet i GrønREFORM, hvor ny kernekraft i DK1 og DK2 erstatter havvind. Velfærdsåret er 2040.

*`1. opsætning`*

*vælg geografi, år og kørsel*

In [2]:
# 1. opsætning
GEO   = "disaggEA"
print("Kernekraft-id'er:", ", ".join(f"{i} (fra {y})" for i, y in use_geography(GEO).items()))

yr    = 2040
ids   = NUCLEAR_IDS_BY_GEO[GEO]
RUN   = "disaggEA_2040_kf25om"           # 2040 med brintfiks
SHOCK = "shock_nuc_2040.pkl"
# RUN   = "disaggEA_2025_2050"           # DREAMs 2025-2050 uden brintfiks

Kernekraft-id'er: idNUC_DK1 (fra 2035), idNUC_DK2 (fra 2035)


*`2. data`*

*indlæs basis- og stødforløb og vis kapacitetsbyttet*

In [3]:
# 2. data
baseline_output, shock_output = load_run(RUN, SHOCK)
nuc, cut = nuc_wind_swap(baseline_output, shock_output, yr=yr, nuc_ids=ids)

C:\Users\LinusLindquist\anaconda3\Lib\site-packages\gams\core\numpy\gams2numpy.py:72: UserWarning: The GAMS version (49.4.0) differs from the API version (54.2.2).
  ws = GamsWorkspace()


Kapacitetsbytte — 2040 (GW)
                    basis  stød      Δ
──────────────────────────────────────
Kernekraft DK1       0,00  0,65  +0,65
Kernekraft DK2       0,00  0,35  +0,35
Havvind DK1          4,50  3,22  -1,28
Havvind DK2          1,79  1,25  -0,55
──────────────────────────────────────
Kernekraft i alt                 +1,00
Havvind i alt                    -1,82
──────────────────────────────────────
Havvind fjernet pr. GW kernekraft: 1,82



*byg timebalancerne pr. zone og for Danmark samlet*

In [4]:
ctx_el     = build_el_balance_data(baseline_output, shock_output, years=yr)
ctx_el_agg = build_el_balance_data_country(baseline_output, shock_output, years=yr)
shares     = build_supply_shares(ctx_el, aggregates=c_aggs_countries)

*`3. priser og knaphed`*

*efterspørgselsvægtede prismomenter og andelen af nulpris- og mangeltimer*

In [5]:
# 3. priser og knaphed
print_price_summary(ctx_el, eas=("DK1", "DK2"), countries=("DK",), years=yr)

Priser og knaphed — 2040
                                Pris (kr./MWh)             Fordeling        Timer (pct.)
                          gns.  min.    maks.  std.afv.  skævhed  kurtose  nulpris  mangel
──────────────────────────────────────────────────────────────────────────────────────────
Danmark (Vest), basis    383,8  40,4  1.179,9     277,2     1,19     0,48      0,0    13,7
Danmark (Vest), stød     382,8  40,4  1.165,5     271,6     1,18     0,47      0,0    13,7
──────────────────────────────────────────────────────────────────────────────────────────
Danmark (Øst), basis     381,0  35,4  1.184,5     271,4     1,21     0,54      0,0    15,9
Danmark (Øst), stød      378,2  35,4  1.157,3     264,7     1,20     0,55      0,0    13,7
──────────────────────────────────────────────────────────────────────────────────────────
Danmark, basis           382,9  38,7  1.169,1     274,9     1,20     0,50      0,0    13,7
Danmark, stød            381,2  38,7  1.146,2     269,0     1,19   

*`4. handel`*

*eksport, import og nettoeksport pr. handelspartner for DK, NO og SE*

In [6]:
# 4. handel
print_trade_summary(ctx_el, countries=("DK", "NO", "SE"), years=yr)

Handel — Danmark, 2040
                           Eksport (TWh)  Import (TWh)      Netto (TWh)        Netto (mio. kr.)
                            basis    stød  basis   stød  basis   stød      Δ   basis    stød     Δ
──────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg      16,51   16,45   4,48   3,93  12,03  12,52  +0,49   3.363   4.113  +750
Storbritannien og Irland     3,75    3,82   4,00   4,14  -0,25  -0,32  -0,07    -397    -307   +91
Nederlandene                 1,07    1,10   1,50   1,59  -0,43  -0,48  -0,05     -82     -18   +64
Norge                        1,53    1,43   8,79   8,88  -7,26  -7,45  -0,19  -1.499  -1.535   -37
Sverige                      3,03    2,98   7,64   7,66  -4,61  -4,69  -0,08  -2.363  -2.332   +31
──────────────────────────────────────────────────────────────────────────────────────────────────
I alt                       25,89   25,78  26,41  26,20  -0,52  -0,42  +0,10    -979     

Handel — Norge, 2040
                           Eksport (TWh)  Import (TWh)      Netto (TWh)       Netto (mio. kr.)
                            basis    stød  basis   stød  basis   stød      Δ  basis   stød    Δ
───────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg       9,08    9,07   1,56   1,57   7,52   7,50  -0,02  1.560  1.559   -1
Danmark                      8,79    8,88   1,53   1,43   7,26   7,45  +0,19  1.499  1.535  +37
Storbritannien og Irland     9,19    9,19   1,49   1,49   7,70   7,69  -0,01  1.418  1.419   +0
Nederlandene                 4,44    4,43   0,86   0,86   3,58   3,57  -0,01    721    721   -1
Nordsøen                     0,00    0,00   1,66   1,67  -1,66  -1,67  -0,01   -299   -302   -3
Sverige                     21,41   21,34   3,48   3,49  17,93  17,86  -0,07  2.993  2.983  -10
───────────────────────────────────────────────────────────────────────────────────────────────
I alt               

Handel — Sverige, 2040
                         Eksport (TWh)  Import (TWh)       Netto (TWh)         Netto (mio. kr.)
                          basis    stød  basis   stød   basis    stød      Δ   basis    stød    Δ
─────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg     9,83    9,84   2,96   2,94    6,87    6,90  +0,03   3.187   3.210  +23
Danmark                    7,64    7,66   3,03   2,98    4,61    4,69  +0,08   2.363   2.332  -31
Baltikum                   2,86    2,87   1,26   1,28    1,60    1,58  -0,02     908     904   -3
Finland                    5,11    5,12  19,37  19,39  -14,25  -14,28  -0,03  -1.681  -1.688   -6
Norge                      3,48    3,49  21,41  21,34  -17,93  -17,86  +0,07  -2.993  -2.983  +10
Nordsøen                   0,00    0,00   3,62   3,62   -3,62   -3,62  +0,00    -960    -957   +3
Polen                      3,28    3,27   0,25   0,26    3,03    3,02  -0,01   1.184   1.180   -4

*bytteforhold pr. handelspartner: eksportpris over importpris*

In [7]:
print_terms_of_trade_partner(ctx_el, country="DK", years=yr)

Bytteforhold pr. handelspartner — Danmark, 2040
                           Eksport (TWh)  Import (TWh)  Eksportpris   Importpris          Bytteforhold
                            basis    stød  basis   stød  basis  stød  basis  stød  basis   stød       Δ  Δ pct.
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
Tyskland og Luxembourg      16,51   16,45   4,48   3,93    325   340    446   375  0,728  0,906  +0,179   +24,6
Storbritannien og Irland     3,75    3,82   4,00   4,14    285   297    367   348  0,778  0,853  +0,076    +9,7
Nederlandene                 1,07    1,10   1,50   1,59    413   455    350   326  1,182  1,395  +0,213   +18,1
Norge                        1,53    1,43   8,79   8,88    124   125    192   193  0,646  0,646  -0,000    -0,0
Sverige                      3,03    2,98   7,64   7,66    119   119    356   351  0,333  0,340  +0,007    +2,0
─────────────────────────────────────────────────────────────────

*`5. fangstpris og værdifaktor`*

*produktionsvægtet fangstpris pr. anlægstype i Danmark*

In [8]:
# 5. fangstpris og værdifaktor
print_capture(capture_summary(ctx_el, country="DK", years=yr), year=yr)

Fangstpris og værdifaktor, el — Danmark, 2040
                                         Produktion (TWh)     Fangstpris (kr./MWh)   Værdifaktor
                                        basis   stød      Δ    basis     stød      Δ            Δ
─────────────────────────────────────────────────────────────────────────────────────────────────
Havvind                                 41,01  33,04  -7,98    298,8    299,2   +0,4        +0,00
Solceller                               19,01  19,03  +0,02    189,0    187,5   -1,5        -0,00
Landvind                                13,98  14,00  +0,01    275,5    278,7   +3,2        +0,01
Modtryksværk (kraftvarme)                1,61   1,60  -0,00    557,7    550,7   -7,0        -0,01
Ellagring                                1,51   1,51  -0,00    655,7    645,7   -9,9        -0,02
Modtryksværk med bypass (kraftvarme)     0,44   0,43  -0,01    818,5    804,7  -13,8        -0,03
Udtagsværk (kraftvarme)                  0,40   0,39  -0,01    884,4    8

*`6. kapacitetsfaktorer`*

*realiseret produktion over installeret kapacitet gange 8760 timer*

In [9]:
# 6. kapacitetsfaktorer
print_capacity_factor(capacity_factor_summary(ctx_el, country="DK", years=yr), year=yr)

Kapacitetsfaktor, el — Danmark, 2040
                                         Produktion (TWh)   Kapacitetsfaktor (pct., ECap)
                                        basis   stød      Δ      basis      stød         Δ
──────────────────────────────────────────────────────────────────────────────────────────
Havvind                                 41,01  33,04  -7,98       50,4      50,5      +0,1
Solceller                               19,01  19,03  +0,02       12,1      12,1      +0,0
Landvind                                13,98  14,00  +0,01       27,8      27,8      +0,0
Modtryksværk (kraftvarme)                1,61   1,60  -0,00       26,9      26,8      -0,1
Ellagring                                1,51   1,51  -0,00        7,6       7,6      -0,0
Modtryksværk med bypass (kraftvarme)     0,44   0,43  -0,01       16,1      15,8      -0,3
Udtagsværk (kraftvarme)                  0,40   0,39  -0,01       13,4      13,1      -0,3
Eksogen elproduktion                     0,06   0,06  

C:\Users\LinusLindquist\e-tank\E-Tank - Data\PROJEKTER\04 KERNEKRAFT\Grøn Reform beregninger og følgegruppe\GrønREFORM_2.0\KF26_wrapper\ET_pylib\market.py:299: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.26]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mult[is_ex] = (1.0 + m.loc[is_ex, "Cv"] / m.loc[is_ex, "Cm"]).to_numpy()


*`7. forbrugeroverskud`*

*ΔCS pr. zone som trapez på DC-IIA-efterspørgslen, $\Delta CS = -\sum_h \left[D_{0h} + \tfrac{1}{2}(D_{1h} - D_{0h})\right]\Delta p_h$*

In [10]:
# 7. forbrugeroverskud
print_cs_table(baseline_output, shock_output, year=yr)

Forbrugeroverskud, el — 2040 (mio. kr.)
                                 heraf
                         ΔCS    pris  mængde
────────────────────────────────────────────
Danmark (Vest)         +44,1   +44,0    +0,1
Danmark (Øst)          +58,3   +58,3    +0,1
Danmark (Bornholm)      +0,0    +0,0    +0,0
Danmark i alt         +102,4  +102,3    +0,1
────────────────────────────────────────────
Norge (syd)            -62,5   -62,5    +0,0
Norge (nord)            -7,2    -7,2    +0,0
Norge (midt)           -12,9   -12,9    +0,0
Norge i alt            -82,6   -82,6    +0,0
────────────────────────────────────────────
Sverige (nord)          -3,7    -3,7    +0,0
Sverige (midtnord)      -6,7    -6,7    +0,0
Sverige (midtsyd)      -28,4   -28,4    +0,0
Sverige (syd)          +14,2   +14,2    +0,0
Sverige i alt          -24,6   -24,6    +0,0
────────────────────────────────────────────
I alt                   -4,8    -4,9    +0,1
────────────────────────────────────────────



*`8. producentoverskud`*

*UP gange kapacitet pr. anlægstype, opdelt i pris-, mængde- og resteffekt*

In [11]:
# 8. producentoverskud
areas = c_aggs["DK"]
maps = dict(id2Pt=baseline_output["id2Pt"].vals, id2EA=shock_output["id2EA"].vals)
dec_maps = dict(**maps,
                id2HA=shock_output["id2HA"].vals,
                id2cIdxE=shock_output["id2cIdxE"].vals,
                id2cIdxH=shock_output["id2cIdxH"].vals)

bt = producer_profit(baseline_output, shock_output, areas=areas, year=yr,
                     split_nuclear=True, **maps)
dec = profit_decomposition(baseline_output, shock_output, areas=areas, year=yr,
                           split_nuclear=True, **dec_maps)
print_producer_profit(bt, year=yr, areas=areas, decomp=dec)

Producentoverskud — Danmark, 2040 (mio. kr.)
                                            Producentoverskud                Δ, heraf
                                           basis      stød         Δ   pris  mængde      rest
─────────────────────────────────────────────────────────────────────────────────────────────
Havvind                                  6.962,5   5.630,4  -1.332,0  +91,3   -11,6  -1.411,7
Solceller                                3.134,9   3.099,5     -35,4  -36,6    +0,1      +1,1
Ellagring                                  510,9     487,8     -23,2  -23,7    +0,9      -0,3
Modtryksværk (kraftvarme)                  205,9     197,4      -8,5   -9,8    +0,2      +1,1
Udtagsværk (kraftvarme)                    126,3     118,8      -7,5   -7,5    +0,1      -0,0
Modtryksværk med bypass (kraftvarme)      -182,4    -188,0      -5,5   -6,7    +0,1      +1,1
Eksogen elproduktion                        11,8      11,0      -0,8   -0,9    +0,0      +0,1
Varmelagring           

*`9. produktion`*

*elproduktion pr. anlægstype i Danmark*

In [12]:
# 9. produktion
by_qty = producer_generation(baseline_output, shock_output, areas=areas, year=yr,
                             split_nuclear=True, **maps)
print_producer_qty(by_qty, year=yr, areas=areas)

Elproduktion — Danmark, 2040 (TWh)
                                        basis   stød      Δ
───────────────────────────────────────────────────────────
Havvind                                 41,01  33,04  -7,98
Udtagsværk (kraftvarme)                  0,40   0,39  -0,01
Modtryksværk med bypass (kraftvarme)     0,44   0,43  -0,01
Modtryksværk (kraftvarme)                1,61   1,60  -0,00
Eksogen elproduktion                     0,06   0,06  -0,00
Kondensværk                              0,00   0,00  -0,00
Vandkraft (flodkraft)                    0,02   0,02  +0,00
Ellagring                                4,20   4,21  +0,01
Landvind                                13,98  14,00  +0,01
Solceller                               26,03  26,06  +0,03
Kernekraft                               0,00   8,01  +8,01
───────────────────────────────────────────────────────────
I alt                                   87,75  87,81  +0,06



*`10. flaskehalsindtægt`*

*flaskehalsindtægt til de danske TSO'er; hver linjes rente deles ligeligt mellem de to ender*

In [13]:
# 10. flaskehalsindtægt
print_ci(baseline_output, shock_output, year=yr, country="DK")

Flaskehalsindtægt til TSO — Danmark, 2040 (mio. kr.)
                        basis     stød      Δ
─────────────────────────────────────────────
Danmark (Vest)        2.060,3  2.015,1  -45,2
Danmark (Øst)           402,2    386,6  -15,6
Danmark (Bornholm)      185,6    182,1   -3,5
─────────────────────────────────────────────
I alt                 2.648,1  2.583,8  -64,3



*`11. velfærd`*

*værdien af kernekraftens fleksibilitet: værkerne tvinges op på minimumslast, og den ekstra produktion værdisættes til modellens priser*
$$
    \Delta W = \sum_{h\in\mathcal H} (p_h-MC-s) \cdot \Delta q_h
$$

In [14]:
# 11. velfærd
print_flex(shock_output, year=yr, u_mins=(1.0, 0.3))      # must-run og 30 pct. minimumslast

Fleksibilitetsværdi, kernekraft — 2040, minimumslast 100 pct.
                        TWh            kr./MWh     mio. kr.
                  produktion  ekstra  pris, ekstra        ΔW
────────────────────────────────────────────────────────────
Kernekraft DK1          5,20    0,49          70,0     -32,7
Kernekraft DK2          2,81    0,26          67,3     -17,9
────────────────────────────────────────────────────────────
I alt                   8,01    0,75          69,1     -50,6
────────────────────────────────────────────────────────────

Fleksibilitetsværdi, kernekraft — 2040, minimumslast 30 pct.
                        TWh            kr./MWh     mio. kr.
                  produktion  ekstra  pris, ekstra        ΔW
────────────────────────────────────────────────────────────
Kernekraft DK1          5,20    0,10          62,2      -7,6
Kernekraft DK2          2,81    0,05          56,9      -4,2
────────────────────────────────────────────────────────────
I alt                   

*samlet velfærdseffekt for Danmark ved tre rentestrukturer*

In [15]:
blocks = welfare_block_by_rate(
    baseline_output, shock_output,
    {"FM": "fm", "4 pct.": 0.04, "6 pct.": 0.06},
    year=yr, regions=("DK",), markets=("E",),
)
print_welfare_blocks(blocks, regions="DK")

Velfærdseffekt — Danmark, 2040 (mio. kr./år)
                                       FM     4 pct.     6 pct.
───────────────────────────────────────────────────────────────
Δ Forbrugeroverskud               +102,40    +102,40    +102,40
Δ Producentoverskud               +113,83    +113,83    +113,83
Δ Flaskehalsindtægter              -64,30     -64,30     -64,30
Annuitet, netto                   -654,80    -885,98  -1.677,82
Systemomkostning, VE            +1.150,68  +1.150,68  +1.150,68
Systemomkostning, kernekraft      -245,03    -245,03    -245,03
───────────────────────────────────────────────────────────────
Netto velfærdseffekt              +402,79    +171,60    -620,24
───────────────────────────────────────────────────────────────
Annuitet, kernekraft (ny)       +2.504,38  +2.853,22  +4.149,17
Annuitet, havvind (nedlagt)     -1.849,58  -1.967,24  -2.471,34
───────────────────────────────────────────────────────────────
Annuitet: + = omkostning, - = sparet.
Systemomkostning, VE:

*`12. LCOE`*

*LCOE af GR's egne drifts- og produktionstal plus annuitet af investeringen*

In [16]:
# 12. LCOE
print_lcoe(shock_output, year=yr, r="fm")

LCOE — Danmark, 2040, rente: fm (øre/kWh)
               kap.faktor (pct.)  annuitet   FOM    MC  LCOE
────────────────────────────────────────────────────────────
Kernekraft                  91,5      31,3  12,5  10,7  54,4
Havvind                     50,5      22,9   9,0   4,0  35,9
Landvind                    27,8      21,6   5,0   1,5  28,2
Solceller                   12,2      19,2   7,9   0,0  27,0
────────────────────────────────────────────────────────────
VE (vægtet)                 22,2      21,3   7,8   2,1  31,3
────────────────────────────────────────────────────────────



C:\Users\LinusLindquist\e-tank\E-Tank - Data\PROJEKTER\04 KERNEKRAFT\Grøn Reform beregninger og følgegruppe\GrønREFORM_2.0\KF26_wrapper\ET_pylib\costs.py:304: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  el_fac = (1 + sym_year(db, "Cv", year).div(sym_year(db, "Cm", year))).reindex(ids).fillna(1.0)
